In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import transforms, models
from PIL import Image
from tqdm import tqdm
import pandas as pd

In [2]:
# Settings for Vgg19 optimized style transfer
# ==========================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'✅ Device: {DEVICE}')

# Paths
REG_PHOTOS_DIR = r"reg_ph_20"  # Regular photos directory
VAN_GOGH_DIR = r"van_gogh_ph_5"  # Van gogh photos directory
OUTPUT_DIR = "final_results_vgg_only"

# Judges paths
JUDGE_VGG_PATH = "best_vangogh_vgg19_final.pth"
JUDGE_ALEX_PATH = "best_vangogh_alexnet_final.pth"

# Best known hyper parameter for a VGG19 optimized style transfer
STYLE_WEIGHT = 651878166.2897693
LR = 0.08144127919297299
TV_WEIGHT = 1.134886253432539e-05
CONTENT_LAYER = 'conv3_2'
IMG_SIZE = 224
NUM_STEPS = 300


LAYER_WEIGHTS = {
    'conv1_1': 0.9867141098321527,
    'conv2_1': 0.7008762804179118,
    'conv3_1': 0.14241736383784193,
    'conv4_1': 0.749579106254155,
    'conv5_1': 0.10321802295397703
}


os.makedirs(OUTPUT_DIR, exist_ok=True)

✅ Device: cuda


In [3]:
# Loading models
# ==========================================

#  VGG Extractor
print("🎨 Loading VGG19 Extractor...")
vgg_extractor = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1).features.to(DEVICE).eval()
for p in vgg_extractor.parameters(): p.requires_grad_(False)


# B. Judges
def load_classifier(arch, path):
    print(f"⚖️ Loading {arch} Judge from {path}...")
    if arch == 'vgg':
        model = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1)
        model.classifier[6] = nn.Linear(4096, 2)
    elif arch == 'alexnet':
        model = models.alexnet(weights=models.AlexNet_Weights.IMAGENET1K_V1)
        model.classifier[6] = nn.Linear(4096, 2)

    try:
        model.load_state_dict(torch.load(path, map_location=DEVICE))
    except FileNotFoundError:
        print(f"⚠️ Warning: Model file {path} not found! Scores for this judge will be 0.")
        return None

    model.to(DEVICE).eval()
    return model


judge_vgg = load_classifier('vgg', JUDGE_VGG_PATH)
judge_alex = load_classifier('alexnet', JUDGE_ALEX_PATH)

🎨 Loading VGG19 Extractor...
⚖️ Loading vgg Judge from best_vangogh_vgg19_final.pth...
⚖️ Loading alexnet Judge from best_vangogh_alexnet_final.pth...


In [4]:
# Loading images and features
# ==========================================
def load_image_tensor(img_path):
    try:
        image = Image.open(img_path).convert('RGB')
    except Exception as e:
        print(f"Error loading {img_path}: {e}")
        return None

    in_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
    ])
    return in_transform(image).unsqueeze(0).to(DEVICE)


def save_tensor_image(tensor, path):
    image = tensor.cpu().clone().detach().squeeze(0)
    inv_normalize = transforms.Normalize(
        mean=[-0.485 / 0.229, -0.456 / 0.224, -0.406 / 0.225],
        std=[1 / 0.229, 1 / 0.224, 1 / 0.225]
    )
    image = inv_normalize(image).clamp(0, 1)
    image = transforms.ToPILImage()(image)
    image.save(path)


def get_features(image, model, layers_dict):
    features = {}
    x = image
    for name, layer in model._modules.items():
        x = layer(x)
        if name in layers_dict:
            features[layers_dict[name]] = x
    return features


def gram_matrix(tensor):
    b, d, h, w = tensor.size()
    tensor = tensor.view(d, h * w)
    return torch.mm(tensor, tensor.t())

In [5]:
# Style Transfer (VGG Specific)
# ==========================================
def run_style_transfer_vgg(content_tensor, style_tensor):
    target = content_tensor.clone().requires_grad_(True).to(DEVICE)
    optimizer = optim.Adam([target], lr=LR)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=100, gamma=0.5)

    # Mapping layers
    layers_map = {
        '0': 'conv1_1', '5': 'conv2_1', '10': 'conv3_1', '12': 'conv3_2',
        '19': 'conv4_1', '21': 'conv4_2', '28': 'conv5_1', '30': 'conv5_2'
    }

    content_features = get_features(content_tensor, vgg_extractor, layers_map)
    style_features = get_features(style_tensor, vgg_extractor, layers_map)
    style_grams = {k: gram_matrix(v) for k, v in style_features.items() if k in LAYER_WEIGHTS}

    for step in range(NUM_STEPS):
        target_features = get_features(target, vgg_extractor, layers_map)

        # Content Loss
        c_loss = torch.mean((target_features[CONTENT_LAYER] - content_features[CONTENT_LAYER]) ** 2)

        # Style Loss
        s_loss = 0
        for layer_name, weight in LAYER_WEIGHTS.items():
            if layer_name in target_features:
                target_gram = gram_matrix(target_features[layer_name])
                style_gram = style_grams[layer_name]
                b, d, h, w = target_features[layer_name].shape
                s_loss += (weight * torch.mean((target_gram - style_gram) ** 2)) / (d * h * w)

        # TV Loss
        diff_i = torch.sum(torch.abs(target[:, :, :, 1:] - target[:, :, :, :-1]))
        diff_j = torch.sum(torch.abs(target[:, :, 1:, :] - target[:, :, :-1, :]))
        tv_loss = (diff_i + diff_j) / target.nelement()

        total_loss = c_loss + STYLE_WEIGHT * s_loss + TV_WEIGHT * tv_loss

        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()
        scheduler.step()

    return target

In [6]:
# Main
# ==========================================
def main():
    reg_images = [f for f in os.listdir(REG_PHOTOS_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    vg_images = [f for f in os.listdir(VAN_GOGH_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    reg_images = reg_images[:20]
    vg_images = vg_images[:5]

    print(f"🚀 Processing {len(reg_images)} Content Images...")
    print(f"   Searching best style out of {len(vg_images)} options per image.")

    results_data = []

    for img_name in tqdm(reg_images, desc="Generating Images"):
        content_path = os.path.join(REG_PHOTOS_DIR, img_name)
        content_tensor = load_image_tensor(content_path)
        if content_tensor is None: continue

        best_img = None
        best_score_by_vgg = -1
        best_style_name = "None"

        # Using all 5 style images
        for style_name in vg_images:
            style_tensor = load_image_tensor(os.path.join(VAN_GOGH_DIR, style_name))

            # Image generating
            gen = run_style_transfer_vgg(content_tensor, style_tensor)

            # VGG judge score
            with torch.no_grad():
                score = 0
                if judge_vgg:
                    score = F.softmax(judge_vgg(gen), dim=1)[0, 1].item()

            # Best score style image
            if score > best_score_by_vgg:
                best_score_by_vgg = score
                best_img = gen
                best_style_name = style_name


        out_path = f"{OUTPUT_DIR}/vgg_{img_name}"
        save_tensor_image(best_img, out_path)

        # Evaluation by the 2 judges
        with torch.no_grad():
            score_vgg = 0
            score_alex = 0
            if judge_vgg:
                score_vgg = F.softmax(judge_vgg(best_img), dim=1)[0, 1].item()
            if judge_alex:
                score_alex = F.softmax(judge_alex(best_img), dim=1)[0, 1].item()

        # Appending final results
        results_data.append({
            "Image": img_name,
            "Best_Style_Used": best_style_name,
            "Generator": "VGG19",
            "Judge_VGG_Score": round(score_vgg, 4),
            "Judge_AlexNet_Score": round(score_alex, 4)
        })

    # Writing the final report file
    df = pd.DataFrame(results_data)
    csv_path = f"{OUTPUT_DIR}/vgg_results_report.csv"
    df.to_csv(csv_path, index=False)

    print("\n" + "=" * 50)
    print(f"🏁 DONE! Images saved in {OUTPUT_DIR}")
    print(f"📄 Report saved to {csv_path}")
    print("=" * 50)
    print("Average Scores:")
    print(df[['Judge_VGG_Score', 'Judge_AlexNet_Score']].mean())
    print("\nStyle Usage Count:")
    print(df['Best_Style_Used'].value_counts())


if __name__ == "__main__":
    main()

🚀 Processing 20 Content Images...
   Searching best style out of 5 options per image.


Generating Images: 100%|██████████| 20/20 [05:32<00:00, 16.64s/it]


🏁 DONE! Images saved in final_results_vgg_only
📄 Report saved to final_results_vgg_only/vgg_results_report.csv
Average Scores:
Judge_VGG_Score        0.933130
Judge_AlexNet_Score    0.381485
dtype: float64

Style Usage Count:
Best_Style_Used
vincent-van-gogh_dr-paul-gachet-1890(1).jpg    20
Name: count, dtype: int64
